# Polynomial Regression
In this exercise, you will implement a simple model to perform a polynomial regression on synthetic data. Recall that in regression you are trying to predict the value of a *target variable* $y$, which is a noisy function of the input:
$$y = f(\mathbf{x}, \mathbf{w}) + \eta.$$
Here, $\mathbf{w}$ represent the parameters of our model, which we want to optimize: for this goal, all we have access to will be a dataset of input-observation pairs, $\mathcal{D} = \{(\mathbf{x}_1, y_1), \dots, (\mathbf{x}_N, y_N)\}$.

In general, we will assume that the noise is distributed according to a zero mean Gaussian:
$$\eta \sim \mathcal{N}(0, \sigma^2).$$
This means that $y$ will be a random variable with distribution
  $$p(y \mid \mathbf{x}, \mathbf{w}, \sigma^2) = \mathcal{N}(y \mid f(\mathbf{x}, \mathbf{w}), \sigma^2).$$

---

## Linear Models
The simplest form for our model is a linear function:
  $$f(\mathbf{x}, \mathbf{w}) = w_0 + w_1 x_1 + \dots + w_D x_D.$$
It is useful to define an artificial coordinate $x_0 = 1$, so that
  $$f(\mathbf{x}, \mathbf{w}) = \mathbf{w}^\intercal \mathbf{x}.$$

If you assume that the observations are also independent, then the likelihood function factorizes as
  $$L(\mathbf{w}) = \prod_{i=1}^N \mathcal{N}(y_i \mid \mathbf{w}^\intercal \mathbf{x}_i, \sigma^2)$$
and you can maximize it with respect to $\mathbf{w}$ to obtain the Maximum Likelihood (ML) estimate for the model parameters: as you have seen in the lectures, the result is
  $$\mathbf{w}_\mathrm{ML} = (\mathbf{X}^\intercal\mathbf{X})^{-1}\mathbf{X}^\intercal \mathbf{y},$$
where $\mathbf{X}$ is called *design matrix* and has the data $\mathbf{x}_i$ as rows, while $\mathbf{y}$ is obtained by stacking up all the observations $y_i$ in a single vector.

## Generalized Linear Models
Linear models are very limited in their expressiveness, as they are only capable of representing linear functions. A solution would be to consider the so-called *generalized linear models*, in which we allow for a (**fixed**!) nonlinear transformation of the data through a set of *basis functions* $\{\phi_i\}$:
  $$f(\mathbf{x}, \mathbf{w}) = \sum_{i=0}^d w_i \phi_i(\mathbf{x}) = \mathbf{w}^\intercal \boldsymbol{\phi}(\mathbf{x}).$$
Since the model is still linear in the parameters, the ML estimate is basically the same (we will see this in the exercise class), thus
  $$\mathbf{w}_\mathrm{ML} = (\boldsymbol{\Phi}^\intercal\boldsymbol{\Phi})^{-1}\boldsymbol{\Phi}^\intercal \mathbf{y},$$
where the design matrix takes now the form
  $$\boldsymbol{\Phi} = \begin{pmatrix} \phi_0(\mathbf{x}_1) & \phi_1(\mathbf{x}_1) & \dots & \phi_d(\mathbf{x}_1) \\
  \phi_0(\mathbf{x}_2) & \phi_1(\mathbf{x}_2) & \dots & \phi_d(\mathbf{x}_2) \\
  \vdots & \vdots & \ddots & \vdots \\
  \phi_0(\mathbf{x}_N) & \phi_1(\mathbf{x}_N) & \dots & \phi_d(\mathbf{x}_N)\end{pmatrix}.$$
Note that, if you choose a particular generalized linear model for modelling your data, you will have to decide on how many basis functions to use. If $d$ is too low, then your model will not have enough power to correctly capture how the data behave; if $d$ is too high, your model will become too sensitive to noise and impossible to use on other data. We will see more about this later and in the next lecture.

### Polynomial Basis Functions
Perhaps the simplest choice we can make for our basis functions is to choose powers of $x$, i.e.
  $$\phi_j(x) = x^j$$
(note for simplicity we will be considering a one-dimensional input from now on). Then the model is a polynomial, the parameters being the coefficients of the different powers of $x$:
$$f(x, \mathbf{w}) = w_0 + w_1 x^1 + \dots + w_d x^d.$$
This is the model you will implement in this exercise; before going on, let's run some standard imports which will be needed for this exercise.

In [14]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from numpy.polynomial.polynomial import Polynomial

# Task 1: Generating Data
First of all, you need some data to work with: you will generate them according to a polynomial law (plus some noise, of course). For this exercise, then, we are cheating by knowing in advance which is the correct $d$ (the polynomial degree). You will not be so lucky with real-world data 🙂

For this task, you'll implement a `polynomial_generator` function, which you will then use to obtain `X, y` samples, where `X` contains all the inputs and `y` contains all the observations.

**Hints**
- For our purposes, it is useful that neither the polynomial degree nor the number of points are too high: stick with $d = 3$ and $n = 20$ respectively
- The function should take as input the number of samples you want to generate, the polynomial degree, and the noise level
- To avoid scaling problems, `X` should be contained in the $[-1, +1]$ interval; also, let the coefficients lie in the $[-2, +2]$ range
- To work with polynomials, it is useful to check out the [NumPy's `polynomial` package](https://numpy.org/doc/stable/reference/routines.polynomials-package.html#module-numpy.polynomial), and in particular the `Polynomial` class (see [here](https://numpy.org/doc/stable/reference/routines.polynomials.html) for a quick, minimal reference)
- The function should return the inputs `X`, the corresponding observations `y`, and the unknown coefficients

In [31]:
def polynomial_generator(n, d, noise, seed=42):
    rng = np.random.default_rng(seed)
    coeff = rng.uniform(-2, 2, d + 1)
    poly = Polynomial(coeff)
    X = rng.uniform(-1, 1, n)
    y = poly(X) + rng.normal(0, noise, n)
    return X,y,coeff


X, y, coeff = polynomial_generator(n=20, d=3, noise=0.2)

fig = px.scatter(x=X,y=y, title="Generated Data", symbol_sequence=["circle-open"])
fig.update_layout(width=800, height=500)
fig.show()

## Task 2: Finding the ML Estimate
Now we have some data! Suppose we want to model it with polynomial regression. First of all, we must be able to calculate the best parameters for a given degree. Remember that the maximum likelihood estimate for the weights is given by
  $$\mathbf{w}_\mathrm{ML} = (\boldsymbol{\Phi}^\intercal \boldsymbol{\Phi})^{-1} \boldsymbol{\Phi}^\intercal \mathbf{y}$$
and build the `polynomial_regression` function, which constructs a polynomial with the ML parameters, given a input-observation dataset and a desired polynomial degree.

Then, try to plot the resulting polynomial for increasing degrees, up to $d = n - 1$: what do you notice?

**Hints**
  - You may find it useful to check out [Vandermonde matrices](https://en.wikipedia.org/wiki/Vandermonde_matrix) and the [corresponding function](https://numpy.org/doc/stable/reference/generated/numpy.vander.html) in NumPy
  - In general, the matrix $(\mathbf{A}^\intercal \mathbf{A})^{-1} \mathbf{A}^\intercal$ is called the *(Moore-Penrose) pseudo-inverse* of $\mathbf{A}$: to calculate it, you can use the [`linalg.pinv` function from NumPy](https://numpy.org/doc/stable/reference/generated/numpy.linalg.pinv.html)

In [41]:
def polynomial_regression(X, y, d):
    Phi = np.vander(X, d + 1, increasing=True)
    w_ml = np.linalg.pinv(Phi) @ y
    return Polynomial(w_ml)


d = 3
p = polynomial_regression(X, y, d)

X_check = np.linspace(-1, 1, 500)
y_check = p(X_check)

fig = px.scatter(x=X, y=y, title=f"Regression with polynomial of degree {d}")
fig.update_traces(marker=dict(symbol="circle-open"))
fig.add_scatter(x=X_check, y=y_check, mode="lines")
fig.update_layout(yaxis_range=[0, 4], showlegend=False)
fig.show()

# Task 3: Comparing Models
Now, we can generate the "best" (in the maximum likelihood sense) polynomial of a certain degree which fits our data. But how can we choose the most appropriate model, i.e. the degree itself?

At first sight, it might look like the higher $d$ is, the better the model will be: higher order polynomials allow for more flexibility, the ability to model more complex functions, and an increased expressiveness. For each model, maximizing the likelihood is equivalent to minimizing the *sum-of-squares* error:
$$E(\mathbf{w}) = \frac{1}{2}\sum_{i=1}^n \bigl(y_i - \mathbf{w}^\intercal \boldsymbol{\phi}(x_i)\bigr)^2.$$
As you will see, this derives from the Gaussian noise assumption; however, a usually better choice is the *mean squared error*, in which we just rescale by the dataset size:
  $$\mathrm{MSE} = \frac{1}{N}\sum_{i=1}^n \bigl(y_i - \mathbf{w}^\intercal \boldsymbol{\phi}(x_i)\bigr)^2,$$
which has the nice property that the errors on datasets of different sizes become comparable. Another common choice is the *root mean squared error*, $\mathrm{RMSE} = \sqrt{\mathrm{MSE}}$, which has the same units as the target. In general, depending on your task, you will have to choose the appropriate error function: for this exercise, we will stick with MSE.

To do so, implement a function which calculates the mean squared error on the dataset for a given polynomial, then try out different polynomials degree and compare the error on the dataset for all of them.

**Hints**
- The `calculate_error` function should calculate the mean squared error on a given dataset, for a given polynomial
- Then, for different polynomial degrees, calculate the error on your dataset, and plot it as a function of the polynomial degree. As before, try out values up to the number of points you have in your dataset. How does the error behave? How does this relate to what you noticed in the previous task when plotting the model?

In [ ]:
def calculate_error():
    raise NotImplementedError("Complete this function for Task 3!")

In [28]:
def calculate_error(y_true, y_pred):
    # 1. Obliczamy różnicę (y_i - y_hat)
    error = y_true - y_pred

    # 2. Podnosimy do kwadratu i wyciągamy średnią
    # To realizuje sumę i dzielenie przez N
    mse = np.mean(error**2)

    return mse

errors = []
degrees = range(0, 20)

for d in degrees:
    # 1. Trenujemy model dla aktualnego stopnia d
    model = polynomial_regression(X, y, d)

    # 2. Obliczamy przewidywania (y_pred) dla naszych punktów X
    y_pred = model(X)

    # 3. Liczymy błąd MSE i dodajemy do listy
    err = calculate_error(y, y_pred)
    errors.append(err)
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(degrees), y=errors, name="Errors", mode="lines+markers"))
fig.update_layout(title="Error for models of increasing degree")
fig.update_xaxes(dtick=1)
fig.show()

# Task 4: Overfitting
As you probably noticed in the previous task, as the degree of the polynomial gets bigger, the error on our dataset becomes smaller. However, if you try to plot the resulting polynomial, you will see that it exihibits some crazy oscillations to pass through all the data points, and the corresponding coefficients have extremely high values (in magnitude)!

This means our model is *too* expressive, in the sense that it is able to fit all the data points, but in doing so it is also fitting the noise and thus will not be reliable for making predictions on unseen data. At the same time, intuitively we would like to favor *simpler* models, and such extreme values for the coefficients look unnatural. If this sounds similar to what you have seen in the lecture about Bayesian modelling, you are on the right track - more on this later and in the next classes 🙂

This behaviour is known as *overfitting*, and there are ways to mitigate it. First of all, as you might have guessed, having more data prevents overfitting: with 20 data points, a 19-degree polynomial can provide a perfect fit. If we had more points, the same model would be less prone to overfitting, allowing us to increase the complexity.

Of course, it's not always possible to have more data; then, a good practice is to split the data you have into a *training set*, used to train the model, and a *validation set*, to regulate the model complexity (in this case, the degree $d$). You will now implement this: out of our 20-points dataset, use only 15 to estimate the model parameters, and test the resulting model on the remaining 5. Then, plot again the error as a function of the polynomial degree, both on the training set and the validation set.

**Hints**
- The `split_data` function should take as input the data `X` and the observations `y`, and return a tuple of four elements: the training data and the corresponding observations, and the validation data and the corresponding observations
- Make sure **not** to mix data and observations: when you return the split datasets, to each data point should still correspond the appropriate observation
- Then, use this to test out different models, this time by only training polynomial models of different degrees on the training data, and then plotting the error on both datasets: what do you notice?

In [37]:
def split_data(X, y, hold_out=5, seed=42):
    rng = np.random.default_rng(seed)

    # 1. Tworzymy listę indeksów od 0 do 19
    indices = np.arange(len(X))

    # 2. Mieszamy indeksy, żeby podział był losowy
    rng.shuffle(indices)

    # 3. Wybieramy indeksy do walidacji (pierwsze 5) i treningu (reszta)
    val_idx = indices[:hold_out]
    train_idx = indices[hold_out:]

    # 4. Wycinamy dane na podstawie tych indeksów
    return X[train_idx], X[val_idx], y[train_idx], y[val_idx]

X_train, X_val, y_train, y_val = split_data(X, y)
degrees = range(0, 20)
train_errors, val_errors = [], []

for d in degrees:
    # Model uczy się TYLKO na danych treningowych
    model = polynomial_regression(X_train, y_train, d)

    # Błąd na tym, co model już widział
    train_errors.append(calculate_error(y_train, model(X_train)))

    # Błąd na danych, których model NIE WIDZIAŁ (prawdziwy egzamin)
    val_errors.append(calculate_error(y_val, model(X_val)))
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(degrees), y=train_errors, name="Train", mode="lines+markers"))
fig.add_trace(go.Scatter(x=list(degrees), y=val_errors, name="Validation", mode="lines+markers"))
fig.update_layout(title="Train Versus Validation Error")
fig.update_xaxes(dtick=1)
fig.show()